# Live Plotting

In [ ]:
import numpy as np
from laboneq.simple import *

from laboneq_applications.automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.automation.web_viewer.server import start_web_viewer
from laboneq_applications.experiments import (
    qubit_spectroscopy,
)
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

# Create a demonstration QuantumPlatform for a 4-qubit tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=6)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup (1x PQSC, 1x SHFQC, 1x HDAWG):
setup = qt_platform.setup

# And a 4-qubit tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of four LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

# We connect to the session in emulation mode:
session = Session(setup)
session.connect(do_emulation=True)

In [ ]:
# Define the frequency ranges to be used for the three qubit
# spectroscopy experiments
qs1_qubit_setup = {"frequencies": np.linspace(5.9e9, 6.4e9, 101)}
qs2_qubit_setup = {"frequencies": np.linspace(6.1e9, 6.6e9, 101)}
qs3_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}
qs4_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}
qs5_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}
qs6_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}
qs7_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}
qs8_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}

# For simplicity we will use the same options for each layer
options = {
    "evaluate": True,
    "update": True,
    "count": 2048,
    "active_reset": True,
}

qubit_ids = ["q0", "q1", "q2", "q3", "q4", "q5"]
qs1_parameters = {"workflow_parameters": {q: dict(qs1_qubit_setup) for q in qubit_ids}}
qs1_parameters["options"] = options
qs2_parameters = {"workflow_parameters": {q: dict(qs2_qubit_setup) for q in qubit_ids}}
qs2_parameters["options"] = options
qs3_parameters = {
    "workflow_parameters": {q: dict(qs3_qubit_setup) for q in ["q0", "q1", "q3", "q5"]}
}
qs3_parameters["options"] = options
qs4_parameters = {"workflow_parameters": {q: dict(qs4_qubit_setup) for q in qubit_ids}}
qs4_parameters["options"] = options
qs5_parameters = {"workflow_parameters": {q: dict(qs5_qubit_setup) for q in qubit_ids}}
qs5_parameters["options"] = options
qs6_parameters = {
    "workflow_parameters": {q: dict(qs6_qubit_setup) for q in ["q1", "q2", "q4", "q5"]}
}
qs6_parameters["options"] = options
qs7_parameters = {"workflow_parameters": {q: dict(qs7_qubit_setup) for q in qubit_ids}}
qs7_parameters["options"] = options
qs8_parameters = {"workflow_parameters": {q: dict(qs8_qubit_setup) for q in qubit_ids}}
qs8_parameters["options"] = options

# Combine the parameters for each layer into one dictionary
# that will be passed to the automation
automation_parameters = {
    "Qubit spec 1": qs1_parameters,
    "Qubit spec fine": qs2_parameters,
    "Qubit spec 3": qs3_parameters,
    "Another qubit spec": qs4_parameters,
    "qs5": qs5_parameters,
    "qs6": qs6_parameters,
    "Final spectroscopy": qs7_parameters,
    "Final spectroscopy_final": qs8_parameters,
}

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=automation_parameters,
    name="Live_plot_example",
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="Qubit spec 1",
    depends_on={"root"},
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="Qubit spec fine",
    depends_on={"Qubit spec 1"},
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q3"],
    key="Qubit spec 3",
    depends_on={"Qubit spec fine"},
)
qs4 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="Another qubit spec",
    depends_on={"Qubit spec fine"},
)
qs5 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="qs5",
    depends_on={"Another qubit spec"},
)
qs6 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q1", "q2", "q4"],
    key="qs6",
    depends_on={"Qubit spec fine", "Qubit spec 3"},
)
qs7 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="Final spectroscopy",
    depends_on={"qs6", "Qubit spec 3", "Qubit spec fine"},
)

In [ ]:
start_web_viewer(auto, port=5000)

In [ ]:
auto.add_layer(qs1)
auto.add_layer(qs2)

# Force the node to fail by placing a high threshold for the $r^2$ of the fit
# in the evaluate_experiment task
qs2 = auto.get_layer("Qubit spec fine")
qs2.evaluation_parameters = {"fit_r2_thresholds": {"q2": 1.0}}

In [ ]:
auto.add_layer(qs3)
auto.add_layer(qs4)

In [ ]:
auto.add_layer(qs5)
auto.add_layer(qs6)

In [ ]:
auto.add_layer(qs7)
qs7 = auto.get_layer("Final spectroscopy")
for q in qubit_ids:
    qs7.evaluation_parameters = {"fit_r2_thresholds": {q: 1.0}}

In [ ]:
auto.run_layer("Qubit spec fine", force=True);

In [ ]:
auto.run_layer("Final spectroscopy", force=True);

In [ ]:
qs8 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    qubit_ids,
    key="Final spectroscopy_final",
    depends_on={"root"},
)
auto.add_layer(qs8)

In [ ]:
auto.plot();